# GAN/Diffusion for PCG

**Domain:** Procedural Generation  ·  *recommended addition*  ·  **runnable:** yes

A refresher on using **generative neural models** — GANs and diffusion models — for
**procedural content generation**: learning the *distribution* of good content from examples,
then sampling brand-new content from it. How they work, the knobs that matter, where they
bite, and when classic algorithmic PCG beats them.

## 1. What & Why

Classic PCG (noise, BSP, cellular automata, WFC, grammars) generates content from
**hand-authored rules**. **Learned PCG** flips that: you collect a corpus of *good* content
(levels, tiles, textures, heightmaps, sprites) and train a model to approximate the
**probability distribution** that produced it, then **sample** new artifacts from that
distribution. The two dominant families:

- **GANs (Generative Adversarial Networks).** A **generator** maps random noise → content; a
  **discriminator** learns to tell real content from generated. They train in a minimax game:
  the generator gets better at fooling the discriminator, the discriminator gets better at
  catching it. At equilibrium the generator produces content indistinguishable from the
  training set. Fast to sample (one forward pass), notoriously fiddly to train.
- **Diffusion models.** Define a fixed **forward process** that gradually adds Gaussian noise
  to data until it's pure noise, then train a network to **reverse** it one small step at a
  time. Sampling starts from noise and iteratively denoises into content. Much more stable to
  train and higher quality/diversity than GANs, but sampling costs many network evaluations.

**The problem they solve.** When the "rules" for good content are too subtle to write down —
the aesthetic of a hand-drawn tileset, the playability feel of human-designed Mario levels,
the look of real-world terrain — you'd rather *learn the style from examples* than encode it.
Learned PCG captures that implicit style and lets you generate endless variations in it.

**When to reach for it.**

- You have a **dataset** of exemplar content and want new content *in the same style*.
- The target style is hard to specify with explicit rules (organic textures, art assets, the
  "feel" of curated levels).
- You want **conditional** generation: "a level with this difficulty", "a texture like this
  sketch" (img2img / inpainting / class conditioning).

**When *not* to.** No dataset, or only a handful of examples → use rule-based PCG (it needs no
data). You need **hard guarantees** (every level solvable, every dungeon connected) →
neural samplers give you *plausible* output, not *correct* output; pair them with a
constraint solver or a verify-and-reject loop. Tiny content (a 9×9 dungeon) where a few lines
of WFC or BSP suffice → don't reach for a GPU.

## 2. Mental Model

**GAN = a forger vs. a detective.** The forger (generator) paints fakes from random
inspiration; the detective (discriminator) studies real art and calls out fakes. Each pushes
the other to improve. When the detective can no longer beat a coin flip, the forger's output
passes for real. The catch: if the detective gets too good too fast, the forger gets no useful
signal and gives up (or paints the *one* fake it knows works — **mode collapse**).

**Diffusion = un-dissolving a photo.** Imagine slowly stirring ink into a glass of water until
it's a uniform gray cloud — that's the **forward** process, and it needs no learning (it's
just "add a little Gaussian noise, T times"). Now train a network to run the film *backwards*:
given a slightly noisy image, predict the noise that was added so you can remove it. Stack
those tiny denoising steps and you can start from a pure gray cloud (random noise) and
**reconstitute a brand-new photo** that was never there.

```
DIFFUSION
  data  x0 ──add noise──▶ x1 ──▶ … ──▶ xT ≈ pure noise     (forward: fixed, no training)
        x0 ◀─denoise──── x1 ◀── … ◀── xT                    (reverse: the network learns this)
            train: predict the noise ε that turned x0 into xt

GAN
        z (noise) ──▶ [Generator] ──▶ fake content ┐
                                                    ├─▶ [Discriminator] ──▶ real? / fake?
                          real content ─────────────┘
            train: G fools D, D catches G (minimax)
```

Key contrast: a GAN learns the whole jump from noise→content **in one shot**; diffusion breaks
that impossible jump into **many easy steps**. That decomposition is *why* diffusion trains so
much more stably.

## 3. Key Concepts

- **Latent / noise input `z`.** GANs sample a random vector `z ~ N(0, I)` and decode it to
  content. The latent space is where "variation" lives; interpolating `z` morphs outputs.
- **Generator & Discriminator.** The two GAN networks. The discriminator is a *learned loss
  function* — it tells the generator what "looks real" means, instead of you hand-coding it.
- **Minimax / adversarial loss.** GAN training optimizes opposing objectives. It has no simple
  "loss goes down" curve; both losses oscillate. You judge progress by **sample quality**, not
  the loss number.
- **Mode collapse.** The GAN failure mode: the generator outputs a few (or one) "safe" samples
  that fool D, ignoring the diversity of the real data. Low sample variance is the tell.
- **Forward (diffusion) process `q`.** A fixed Markov chain that adds noise per a **variance
  schedule** `β₁…β_T`. Crucially it has a **closed form**: you can jump straight to any noise
  level `t` with `xₜ = √(ᾱₜ)·x₀ + √(1−ᾱₜ)·ε`, where `ᾱₜ = Π(1−βᵢ)`. No simulation needed
  during training.
- **Reverse process / denoiser.** The network `εθ(xₜ, t)` that **predicts the noise** in a
  noisy sample. Training is plain regression: MSE between predicted and actual noise. Time `t`
  is fed in (as an embedding) so one network handles every noise level.
- **Sampling steps `T`.** Reverse-diffusion takes `T` network calls (50–1000). More steps =
  better quality, slower. **DDIM** and distillation cut this to a handful.
- **Conditioning.** Feed an extra signal — class label, difficulty, a sketch, text — so
  generation is *steerable* (cGAN, classifier-free guidance for diffusion). This is what makes
  learned PCG controllable rather than slot-machine.
- **Representation matters most.** For PCG the hardest choice is *how you encode content* for
  the net: one-hot tile grids, integer maps, signed-distance fields, parameter vectors. The
  encoding decides what the model can express and whether outputs are even valid.

## 4. Setup

Everything here is deliberately tiny and **CPU-only**: small MLPs over 1-D "terrain profiles"
(a row of heights), a few thousand training steps each, seconds to run. Real PCG models are
convolutional and GPU-trained, but the *mechanics* — noise schedule, denoising regression,
adversarial game — are identical at this scale.

`torch` does the modelling; `numpy` builds the toy dataset; `matplotlib` (headless) draws the
samples. No downloads, no API keys.

In [1]:
# %pip install numpy torch matplotlib
import numpy as np
import torch
import torch.nn as nn

torch.manual_seed(0)
np.random.seed(0)
DEVICE = "cpu"  # tiny models; CPU is plenty

print("torch", torch.__version__, "| numpy", np.__version__)

torch 2.12.1 | numpy 2.4.6


## 5. Worked Examples

We treat a **1-D terrain profile** (a length-32 row of heights — think a cross-section of
rolling hills) as our "content". The dataset is a pile of smooth, low-frequency curves; a
*good* sample is smooth and wavy, a *bad* one is jagged white noise. This is small enough to
train in seconds yet rich enough to show learning, mode collapse, and quality metrics.

In [2]:
L = 32  # length of one terrain profile (a row of heights)

def make_terrain(n, L=L, seed=0):
    """Smooth profiles = sum of a few low-frequency sinusoids with random amp/phase."""
    rng = np.random.default_rng(seed)
    x = np.linspace(0, 2 * np.pi, L)
    out = np.zeros((n, L))
    for k in range(1, 4):                       # frequencies 1,2,3 -> "rolling hills"
        amp = rng.normal(0, 1.0 / k, size=(n, 1))
        phase = rng.uniform(0, 2 * np.pi, size=(n, 1))
        out += amp * np.sin(k * x[None, :] + phase)
    out /= np.abs(out).max(axis=1, keepdims=True)  # normalize each profile to ~[-1, 1]
    return out.astype(np.float32)

data = make_terrain(2048)
X = torch.tensor(data)
print("dataset:", data.shape, "| value range",
      round(float(data.min()), 2), "to", round(float(data.max()), 2))

dataset: (2048, 32) | value range -1.0 to 1.0


### Example 1 — the forward diffusion process (no training needed)

The forward process is *fixed*: pick a variance schedule `β₁…β_T`, and you can noise any
sample to level `t` in one step via the closed form
`xₜ = √(ᾱₜ)·x₀ + √(1−ᾱₜ)·ε`. As `t` grows, `ᾱₜ → 0`: the original signal fades and the sample
becomes pure Gaussian noise. This is the "stir ink into water" half — there is nothing to
learn here, and it's exactly what generates the training pairs for the denoiser.

In [3]:
T = 50
betas = torch.linspace(1e-4, 0.15, T)        # variance schedule (steep so 50 steps fully noise it)
alphas = 1.0 - betas
abar = torch.cumprod(alphas, dim=0)          # \bar{alpha}_t = prod(1 - beta_i)

def q_sample(x0, t, noise=None):
    """Forward process in closed form: jump straight to noise level t."""
    if noise is None:
        noise = torch.randn_like(x0)
    a = abar[t].sqrt().unsqueeze(-1)         # signal kept
    b = (1 - abar[t]).sqrt().unsqueeze(-1)   # noise added
    return a * x0 + b * noise

x0 = X[:1]                                    # one terrain profile
print(f"{'t':>3} | {'abar_t':>7} | {'signal_kept':>11} | {'sample_std':>10}")
print("-" * 42)
for t in [0, 10, 25, 49]:
    xt = q_sample(x0, torch.tensor([t]))
    print(f"{t:>3} | {abar[t]:>7.3f} | {abar[t].sqrt():>11.2f} | {xt.std():>10.2f}")
print("\nAs t rises, signal_kept -> 0 and the profile dissolves into unit-variance noise.")

  t |  abar_t | signal_kept | sample_std
------------------------------------------
  0 |   1.000 |        1.00 |       0.52
 10 |   0.843 |        0.92 |       0.56
 25 |   0.359 |        0.60 |       0.99
 49 |   0.019 |        0.14 |       1.04

As t rises, signal_kept -> 0 and the profile dissolves into unit-variance noise.


### Example 2 — train a tiny diffusion model and sample new terrain

The denoiser is a time-conditioned MLP `εθ(xₜ, t)` that predicts the noise added to a sample.
Training is plain MSE regression: take a real profile, noise it to a random `t`, ask the net
for the noise back. To **generate**, start from pure noise and walk the reverse chain,
subtracting the predicted noise a little at a time (DDPM ancestral sampling).

We score quality with **roughness** = mean |second difference|. Real terrain is smooth (low
roughness); white noise is jagged (high). If learning worked, generated samples land near the
real value, far below noise.

In [4]:
class Denoiser(nn.Module):
    def __init__(self, L=L, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(L + 1, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.SiLU(),
            nn.Linear(hidden, L),
        )

    def forward(self, x, t):
        tt = (t.float() / T).unsqueeze(-1)        # feed normalized timestep as a feature
        return self.net(torch.cat([x, tt], dim=-1))

model = Denoiser()
opt = torch.optim.Adam(model.parameters(), lr=2e-3)

losses = []
for step in range(2000):
    idx = torch.randint(0, X.shape[0], (128,))
    x0 = X[idx]
    t = torch.randint(0, T, (128,))
    noise = torch.randn_like(x0)
    xt = q_sample(x0, t, noise)
    pred = model(xt, t)
    loss = ((pred - noise) ** 2).mean()           # predict the noise -> MSE
    opt.zero_grad(); loss.backward(); opt.step()
    losses.append(loss.item())

print(f"loss: {np.mean(losses[:50]):.3f} (start) -> {np.mean(losses[-50:]):.3f} (final)")

loss: 0.665 (start) -> 0.140 (final)


In [5]:
@torch.no_grad()
def ddpm_sample(n):
    """Reverse process: start from noise, denoise step by step into content."""
    x = torch.randn(n, L)
    for t in reversed(range(T)):
        tb = torch.full((n,), t, dtype=torch.long)
        eps = model(x, tb)
        a, ab, b = alphas[t], abar[t], betas[t]
        mean = (x - b / (1 - ab).sqrt() * eps) / a.sqrt()
        x = mean + (b.sqrt() * torch.randn_like(x) if t > 0 else 0.0)
    return x.numpy()

def roughness(arr):                                # mean |2nd difference|: low=smooth, high=noisy
    return float(np.abs(np.diff(arr, n=2, axis=1)).mean())

diff_gen = ddpm_sample(512)
print(f"roughness  real terrain  : {roughness(data):.3f}")
print(f"roughness  pure noise    : {roughness(np.random.randn(512, L)):.3f}")
print(f"roughness  diffusion gen : {roughness(diff_gen):.3f}   <- learned the smooth style")

roughness  real terrain  : 0.069
roughness  pure noise    : 1.927
roughness  diffusion gen : 0.234   <- learned the smooth style


### Example 3 — a tiny GAN for contrast (and a mode-collapse check)

Same data, the adversarial recipe. A generator maps `z ~ N(0,I)` → profile; a discriminator
scores real vs. fake; they train in opposition. GANs sample in **one** forward pass (vs.
diffusion's `T` steps) but are touchier. We check two things: did it learn the smooth style
(roughness), and did it keep **diversity** (per-position std across samples — a low value
versus the real data signals mode collapse).

In [6]:
class Generator(nn.Module):
    def __init__(self, z=8, L=L, h=128):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(z, h), nn.SiLU(),
                                 nn.Linear(h, h), nn.SiLU(),
                                 nn.Linear(h, L), nn.Tanh())
    def forward(self, z): return self.net(z)

class Discriminator(nn.Module):
    def __init__(self, L=L, h=128):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(L, h), nn.LeakyReLU(0.2),
                                 nn.Linear(h, h), nn.LeakyReLU(0.2),
                                 nn.Linear(h, 1))
    def forward(self, x): return self.net(x)

ZDIM = 8
G, D = Generator(ZDIM), Discriminator()
og = torch.optim.Adam(G.parameters(), lr=2e-4, betas=(0.5, 0.9))
od = torch.optim.Adam(D.parameters(), lr=2e-4, betas=(0.5, 0.9))
bce = nn.BCEWithLogitsLoss()
ones, zeros = torch.ones(128, 1), torch.zeros(128, 1)

for step in range(4000):
    real = X[torch.randint(0, X.shape[0], (128,))]
    fake = G(torch.randn(128, ZDIM)).detach()
    od.zero_grad()                                   # train discriminator
    (bce(D(real), ones) + bce(D(fake), zeros)).backward(); od.step()
    og.zero_grad()                                   # train generator (fool D)
    bce(D(G(torch.randn(128, ZDIM))), ones).backward(); og.step()

with torch.no_grad():
    gan_gen = G(torch.randn(512, ZDIM)).numpy()

print(f"roughness        real {roughness(data):.3f} | GAN {roughness(gan_gen):.3f}"
      "   (close => GAN learned the smooth style)")
print(f"per-pos std      real {data.std(0).mean():.3f} | GAN {gan_gen.std(0).mean():.3f}"
      "   (diversity check: a big drop on GAN would signal mode collapse)")

roughness        real 0.069 | GAN 0.124   (close => GAN learned the smooth style)
per-pos std      real 0.563 | GAN 0.525   (diversity check: a big drop on GAN would signal mode collapse)


Render real vs. diffusion vs. GAN samples side by side. Diffusion samples should look like
the real rolling-hill profiles with comparable variety; the GAN's may be smooth but visibly
less diverse if it has started to collapse — watch the diversity metric for a sharp drop.

In [7]:
import matplotlib
matplotlib.use("Agg")  # headless: no display server needed
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(11, 3), sharey=True)
for ax, (title, arr) in zip(axes, [("real", data), ("diffusion", diff_gen), ("GAN", gan_gen)]):
    for row in arr[:8]:
        ax.plot(row, lw=1, alpha=0.7)
    ax.set_title(f"{title} terrain")
    ax.set_xlabel("position")
axes[0].set_ylabel("height")
fig.tight_layout()
fig.savefig("pcg_samples.png", dpi=90)
print("saved pcg_samples.png")

saved pcg_samples.png


## 6. Gotchas & Pitfalls

- **GAN training instability.** The losses don't monotonically decrease — they oscillate, and
  a "good" loss number can mean nothing. Judge by samples. Use the stabilizers everyone uses:
  `Adam(betas=(0.5, 0.9))`, lower LR for the generator, label smoothing, spectral norm or a
  Wasserstein/WGAN-GP loss. Don't expect a vanilla GAN to "just work".
- **Mode collapse.** The generator finds a handful of outputs that fool D and stops exploring.
  Symptoms: low sample diversity, repeated outputs. Mitigate with minibatch discrimination,
  WGAN-GP, or unrolled GANs — or just **use diffusion**, which doesn't have this failure mode.
- **Diffusion is slow to sample.** `T` network calls per sample. For interactive PCG that's a
  killer; reach for **DDIM** (deterministic, 10–50 steps), step distillation, or latent
  diffusion (denoise in a compressed space). Training cost is also non-trivial.
- **Invalid content.** Neural samplers output *plausible* artifacts, not *valid* ones — a
  generated dungeon may be disconnected, a level unbeatable, a tile map full of illegal
  adjacencies. Always **post-validate** (flood-fill connectivity, A* solvability, adjacency
  rules) and reject/repair. Combine with WFC or a constraint solver for hard guarantees.
- **Discrete tiles need care.** Levels are usually *categorical* tile IDs, but nets emit
  continuous values. Use **one-hot channels + argmax** (or Gumbel-softmax), not a single
  integer-regressed channel — regressing "tile 3.7" is meaningless.
- **Tiny datasets overfit / mode-collapse fast.** Most PCG corpora are small (think the few
  hundred hand-made Mario levels). Heavy augmentation (crops, flips, symmetry) and small
  models are essential; otherwise the GAN memorizes a few levels.
- **Resolution & receptive field.** A fully-connected net (like here) doesn't scale to large
  2-D maps — use **convolutions** so structure is translation-invariant and the model sees
  enough context. The toy MLP is for clarity, not production.
- **Evaluation is hard.** "Looks good" isn't a metric. Track concrete proxies — diversity,
  validity rate, playability, distance-to-nearest-training-sample (to catch memorization) —
  the way we used roughness and per-position std above.

## 7. When to Use vs Alternatives

| Approach | Strengths | Weaknesses | Reach for it when |
|---|---|---|---|
| **Diffusion** | Stable training, high quality **and** diversity, great conditioning (inpainting, img2img, text) | Slow sampling (many steps), heavier to train | You have data and want the best quality/diversity; controllable generation |
| **GAN** | One-pass sampling (fast), sharp outputs, mature tooling | Unstable training, mode collapse, weaker diversity | Real-time sampling matters and you can afford to tune training |
| **VAE** | Stable, smooth latent space, easy interpolation | Blurrier outputs than GAN/diffusion | You want a clean latent to interpolate/edit content in |
| **WFC / constraint solvers** | **Hard guarantees** (valid adjacencies), no data needed | Local-only constraints, can contradict & backtrack | Tile maps with strict local rules; correctness over realism |
| **Rule-based PCG** (noise, BSP, CA, grammars) | No data, no training, fast, debuggable, deterministic | You must author the rules; limited to expressible styles | No dataset, small content, or you need full control |

**Rule of thumb:** if you have a **dataset** and the style is hard to write down, learn it —
**diffusion** is the modern default, **GAN** when one-pass sampling speed is critical. If you
have **rules but no data**, stay with classic PCG (see the WFC, Perlin/Simplex, BSP, and
Cellular Automata notebooks in this domain). For shippable content, the strong move is often
**hybrid**: learn the style with a neural model, then enforce hard constraints with WFC or a
verify-and-repair pass on top.

## 8. Resources

- **Ho et al., "Denoising Diffusion Probabilistic Models" (DDPM)** — the paper this notebook's
  diffusion example mirrors: <https://arxiv.org/abs/2006.11239>
- **Lilian Weng, "What are Diffusion Models?"** — the clearest end-to-end derivation of the
  math, with all the schedules and sampling variants:
  <https://lilianweng.github.io/posts/2021-07-11-diffusion-models/>
- **Goodfellow et al., "Generative Adversarial Networks"** — the original GAN paper:
  <https://arxiv.org/abs/1406.2661>
- **Volz et al., "Evolving Mario Levels in the Latent Space of a DCGAN"** — the canonical
  learned-PCG paper (GAN + latent-space search over real game levels):
  <https://arxiv.org/abs/1805.00728>
- **Summerville et al., "Procedural Content Generation via Machine Learning (PCGML)"** — the
  survey that frames the whole field: <https://arxiv.org/abs/1702.00539>
- **The Annotated Diffusion Model (Hugging Face)** — a runnable, line-by-line PyTorch DDPM:
  <https://huggingface.co/blog/annotated-diffusion>